# 🩺 Breast Cancer Multimodal Detection
**Modalities:** Ultrasound · Histopathology · Chest X-ray  
**Architecture:** EfficientNetB0 × 3 → Feature Fusion → Dense(512) → Dense(256) → Sigmoid

---
### ⚠️ GPU Enable karo PEHLE:
**Runtime → Change runtime type → T4 GPU → Save**

### Steps:
1. Step 1 — Install
2. Step 2 — Clone repo
3. Step 3 — Kaggle setup
4. Step 4 — Download datasets
5. Step 5 — Organize datasets
6. Step 6 — Train multimodal model (~30-40 min)
7. Step 7 — Train tabular models
8. Step 8 — Flask app run
9. Step 9 — Ngrok launch

## Step 1 — Install Dependencies
*(Numpy warnings at end are NORMAL — ignore them)*

In [ ]:
!pip install -q flask flask-cors pillow scikit-learn numpy==1.26.4 pandas joblib kaggle pyngrok
print('✅ Done! Numpy warnings upar wale ignore karo — normal hain.')

## Step 2 — Clone GitHub Repo

In [ ]:
import os, sys, shutil

REPO_URL = 'https://github.com/fadicrazy/AI-PROJECT.git'
REPO_DIR = '/content/AI-PROJECT'

# Fresh clone har baar
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

os.system(f'git clone {REPO_URL}')

# Repo directly root mein hai (no breast_cancer_ai subfolder)
PROJ_DIR = REPO_DIR
sys.path.insert(0, PROJ_DIR)
os.chdir(PROJ_DIR)

print('✅ Repo cloned!')
print('Files:', os.listdir(PROJ_DIR))

## Step 3 — Kaggle API Setup
Bas yeh cell run karo — username aur key already set hai.

In [ ]:
import os, json

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

kaggle_data = {
    "username": "fahadshabbirbhatti",
    "key": "KGAT_48ae54a3311198008b51e46c6c4f5108"
}

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(kaggle_data, f)

os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle API configured!')

# Verify
result = os.popen('kaggle datasets list --max-size 1 2>&1').read()
if 'ref' in result or 'title' in result:
    print('✅ Kaggle API working!')
else:
    print('Output:', result[:200])

## Step 4 — Download Datasets
*(~10-15 minutes — chhod do, kuch mat karo beech mein)*

In [ ]:
import os
DATA_ROOT = '/content/datasets'
os.makedirs(DATA_ROOT, exist_ok=True)

print('[1/3] Downloading Ultrasound (BUSI alternative)...')
!kaggle datasets download -d vuppalaadithyasairam/ultrasound-breast-images-for-breast-cancer -p {DATA_ROOT}/raw_us --unzip

print('[2/3] Downloading Histopathology (IDC)...')
!kaggle datasets download -d paultimothymooney/breast-histopathology-images -p {DATA_ROOT}/raw_histo --unzip

print('[3/3] Chest X-ray already downloaded (from previous session or run below)...')
if not os.path.exists(f'{DATA_ROOT}/raw_xray/chest_xray'):
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p {DATA_ROOT}/raw_xray --unzip

print('\n✅ Downloads done!')
print('Datasets:', os.listdir(DATA_ROOT))

## Step 5 — Organize Datasets (train/val/test split)

In [ ]:
import os, shutil, random
from pathlib import Path

DATA_ROOT = '/content/datasets'
random.seed(42)

def split_and_copy(files, label, dest_dir, val_ratio=0.15, test_ratio=0.10):
    random.shuffle(files)
    n = len(files)
    nv, nt = int(n*val_ratio), int(n*test_ratio)
    splits = {'train': files[:n-nv-nt], 'val': files[n-nv-nt:n-nt], 'test': files[n-nt:]}
    for split, imgs in splits.items():
        out = Path(dest_dir)/split/label
        out.mkdir(parents=True, exist_ok=True)
        for img in imgs: shutil.copy(img, out/img.name)
    print(f'  {label}: {len(splits["train"])} train | {len(splits["val"])} val | {len(splits["test"])} test')

# ── Ultrasound ────────────────────────────────────────────────────────────────
print('Organizing Ultrasound...')
us_base = Path(f'{DATA_ROOT}/raw_us/ultrasound breast classification/train')
split_and_copy(
    [f for f in (us_base/'benign').rglob('*') if f.suffix.lower() in ['.png','.jpg','.jpeg']],
    'benign', f'{DATA_ROOT}/ultrasound'
)
split_and_copy(
    [f for f in (us_base/'malignant').rglob('*') if f.suffix.lower() in ['.png','.jpg','.jpeg']],
    'malignant', f'{DATA_ROOT}/ultrasound'
)

# ── Histopathology (IDC) ──────────────────────────────────────────────────────
print('Organizing Histopathology (IDC)...')
all_benign, all_malignant = [], []
histo_root = Path(f'{DATA_ROOT}/raw_histo')
for patient_dir in histo_root.iterdir():
    if patient_dir.is_dir() and patient_dir.name != 'IDC_regular_ps50_idx5':
        if (patient_dir/'0').exists():
            all_benign.extend([f for f in (patient_dir/'0').rglob('*.png')])
        if (patient_dir/'1').exists():
            all_malignant.extend([f for f in (patient_dir/'1').rglob('*.png')])

print(f'  Total found — benign: {len(all_benign)}, malignant: {len(all_malignant)}')
random.shuffle(all_benign); random.shuffle(all_malignant)
split_and_copy(all_benign[:5000],    'benign',    f'{DATA_ROOT}/histopathology')
split_and_copy(all_malignant[:5000], 'malignant', f'{DATA_ROOT}/histopathology')

# ── Chest X-ray ───────────────────────────────────────────────────────────────
print('Organizing Chest X-ray...')
xray_train = Path(f'{DATA_ROOT}/raw_xray/chest_xray/train')
split_and_copy(
    [f for f in (xray_train/'NORMAL').rglob('*') if f.suffix.lower() in ['.png','.jpg','.jpeg']],
    'benign', f'{DATA_ROOT}/chest_xray'
)
split_and_copy(
    [f for f in (xray_train/'PNEUMONIA').rglob('*') if f.suffix.lower() in ['.png','.jpg','.jpeg']],
    'malignant', f'{DATA_ROOT}/chest_xray'
)

print('\n✅ All datasets organized!')

## Step 6 — Train Multimodal Model
⚠️ **GPU zaroori hai!** Runtime → Change runtime type → T4 GPU  
*(~30-40 minutes — chhod do)*

In [ ]:
import os, json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, roc_auc_score

IMG_SIZE   = 224
BATCH_SIZE = 32
DROPOUT    = 0.4
LR         = 1e-4
DATA_ROOT  = '/content/datasets'
PROJ_DIR   = '/content/AI-PROJECT'
MODELS_DIR = os.path.join(PROJ_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

def make_datagen(validation=False):
    if validation:
        return ImageDataGenerator(rescale=1./255)
    return ImageDataGenerator(rescale=1./255, horizontal_flip=True,
        vertical_flip=True, rotation_range=20, zoom_range=0.15,
        width_shift_range=0.1, height_shift_range=0.1, fill_mode='nearest')

def make_generators(data_dir, dg_train, dg_val):
    kw = dict(target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
              class_mode='binary', classes=['benign','malignant'], seed=42)
    return (dg_train.flow_from_directory(os.path.join(data_dir,'train'), **kw),
            dg_val.flow_from_directory(os.path.join(data_dir,'val'), **kw))

def build_extractor(name):
    inp  = layers.Input(shape=(IMG_SIZE,IMG_SIZE,3), name=f'input_{name}')
    base = EfficientNetB0(include_top=False, weights='imagenet',
                          input_tensor=inp, pooling='avg')
    base.trainable = False
    return Model(inputs=inp, outputs=base.output, name=f'efficientnet_{name}')

def build_model():
    i_us    = layers.Input((IMG_SIZE,IMG_SIZE,3), name='input_ultrasound')
    i_histo = layers.Input((IMG_SIZE,IMG_SIZE,3), name='input_histopathology')
    i_xray  = layers.Input((IMG_SIZE,IMG_SIZE,3), name='input_chest_xray')
    e_us    = build_extractor('ultrasound')
    e_histo = build_extractor('histopathology')
    e_xray  = build_extractor('chest_xray')
    fused = layers.Concatenate(name='feature_fusion')(
        [e_us(i_us), e_histo(i_histo), e_xray(i_xray)])
    x = layers.Dense(512, activation='relu')(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    return Model(inputs=[i_us, i_histo, i_xray], outputs=out, name='BreastCancer_Multimodal')

# ── Fixed tf.data generator (no TypeError) ───────────────────────────────────
def make_tf_dataset(g_us, g_histo, g_xray, n_steps):
    def gen():
        for _ in range(n_steps * 2):  # x2 for fine-tune phase
            x1, y = next(g_us)
            x2, _ = next(g_histo)
            x3, _ = next(g_xray)
            bs = min(len(y), len(x2), len(x3))
            for i in range(bs):
                yield (x1[i].astype(np.float32),
                       x2[i].astype(np.float32),
                       x3[i].astype(np.float32)), np.float32(y[i])

    sig = (
        (
            tf.TensorSpec(shape=(IMG_SIZE,IMG_SIZE,3), dtype=tf.float32),
            tf.TensorSpec(shape=(IMG_SIZE,IMG_SIZE,3), dtype=tf.float32),
            tf.TensorSpec(shape=(IMG_SIZE,IMG_SIZE,3), dtype=tf.float32),
        ),
        tf.TensorSpec(shape=(), dtype=tf.float32)
    )
    return (tf.data.Dataset.from_generator(gen, output_signature=sig)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print('Building generators...')
dg_tr, dg_vl = make_datagen(), make_datagen(validation=True)
tr_us,  vl_us  = make_generators(f'{DATA_ROOT}/ultrasound',     dg_tr, dg_vl)
tr_hs,  vl_hs  = make_generators(f'{DATA_ROOT}/histopathology', dg_tr, dg_vl)
tr_cx,  vl_cx  = make_generators(f'{DATA_ROOT}/chest_xray',     dg_tr, dg_vl)

steps = min(tr_us.samples, tr_hs.samples, tr_cx.samples) // BATCH_SIZE
vstep = min(vl_us.samples, vl_hs.samples, vl_cx.samples) // BATCH_SIZE
print(f'Steps: {steps} | Val steps: {vstep}')

train_ds = make_tf_dataset(tr_us, tr_hs, tr_cx, steps)
val_ds   = make_tf_dataset(vl_us, vl_hs, vl_cx, vstep)

print('Building model...')
model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
model.summary()

cbs = [
    EarlyStopping(monitor='val_auc', patience=7,
                  restore_best_weights=True, mode='max', verbose=1),
    ModelCheckpoint(os.path.join(MODELS_DIR,'multimodal_best.keras'),
                    monitor='val_auc', save_best_only=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1)
]

print('\n[Phase 1] Training (EfficientNetB0 frozen)...')
model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs)

print('\n[Phase 2] Fine-tuning top-20 layers...')
for lyr in model.layers:
    if lyr.name.startswith('efficientnet_'):
        lyr.trainable = True
        subs = lyr.layers if hasattr(lyr,'layers') else []
        for i,s in enumerate(subs):
            s.trainable = (i >= len(subs)-20)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR/10),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

model.save(os.path.join(MODELS_DIR,'multimodal_model.keras'))

# Save metrics
metrics = {
    'model': 'Multimodal EfficientNetB0 (US + Histo + CXR)',
    'modalities': ['ultrasound','histopathology','chest_xray'],
    'architecture': {
        'feature_extractor': 'EfficientNetB0 x3 (ImageNet pretrained)',
        'fusion': 'Concatenation (3x1280=3840 dims)',
        'dense_layers': [512, 256],
        'output': 'Sigmoid (Malignant/Benign)'
    }
}
with open(os.path.join(PROJ_DIR,'multimodal_metrics.json'),'w') as f:
    json.dump(metrics, f, indent=2)

print('\n✅ Multimodal model trained and saved!')

## Step 7 — Train Tabular + Original CNN Models

In [ ]:
import os
os.chdir('/content/AI-PROJECT')
!python train_models.py
!python train_image_model.py

## Step 8 — Flask App Setup

In [ ]:
import base64, json, os, random, sys
from io import BytesIO
import joblib, numpy as np, pandas as pd
from flask import Flask, jsonify, render_template, request
from flask_cors import CORS
from PIL import Image
from sklearn.datasets import load_breast_cancer

PROJ_DIR = '/content/AI-PROJECT'
sys.path.insert(0, PROJ_DIR)
os.chdir(PROJ_DIR)

from agents.agent import CancerAgent

MODELS_DIR    = os.path.join(PROJ_DIR, 'models')
TEMPLATES_DIR = os.path.join(PROJ_DIR, 'templates')

app = Flask(__name__, template_folder=TEMPLATES_DIR)
CORS(app, resources={r'/api/*': {'origins': '*'}})
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024

cnn_model        = None
multimodal_model = None
agent            = CancerAgent()

with open(os.path.join(PROJ_DIR, 'model_results.json')) as f:
    MODEL_RESULTS = json.load(f)
with open(os.path.join(PROJ_DIR, 'cnn_metrics.json')) as f:
    CNN_METRICS = json.load(f)

_mm = os.path.join(PROJ_DIR, 'multimodal_metrics.json')
MULTIMODAL_METRICS = json.load(open(_mm)) if os.path.exists(_mm) else {}

SCALER        = joblib.load(os.path.join(MODELS_DIR, 'scaler.pkl'))
FEATURE_NAMES = joblib.load(os.path.join(MODELS_DIR, 'feature_names.pkl'))
MODELS = {
    'Random Forest':       joblib.load(os.path.join(MODELS_DIR, 'random_forest.pkl')),
    'Gradient Boosting':   joblib.load(os.path.join(MODELS_DIR, 'gradient_boosting.pkl')),
    'SVM':                 joblib.load(os.path.join(MODELS_DIR, 'svm.pkl')),
    'Logistic Regression': joblib.load(os.path.join(MODELS_DIR, 'logistic_regression.pkl')),
    'KNN':                 joblib.load(os.path.join(MODELS_DIR, 'knn.pkl')),
}
DATA          = load_breast_cancer(as_frame=True)
DF            = DATA.frame
FEATURE_MEANS = DF[FEATURE_NAMES].mean()
FEATURE_STDS  = DF[FEATURE_NAMES].std()
FEATURE_EXAMPLES = {n: float(round(FEATURE_MEANS[n],3)) for n in FEATURE_NAMES}
TARGET_LABELS = {0: 'Malignant', 1: 'Benign'}
print('✅ Flask app setup done!')

In [ ]:
def lazy_load_cnn():
    global cnn_model
    if cnn_model is None:
        import tensorflow as tf
        cnn_model = tf.keras.models.load_model(os.path.join(MODELS_DIR,'cnn_model.keras'))
    return cnn_model

def lazy_load_multimodal():
    global multimodal_model
    if multimodal_model is None:
        import tensorflow as tf
        p = os.path.join(MODELS_DIR,'multimodal_model.keras')
        if not os.path.exists(p):
            raise FileNotFoundError('Multimodal model not found. Run Step 6 first.')
        multimodal_model = tf.keras.models.load_model(p)
    return multimodal_model

def normalize_features(fv):
    return SCALER.transform(np.array(fv, dtype=float).reshape(1,-1))

def risk_level(p):
    return 'High' if p>=0.85 else ('Moderate' if p>=0.65 else 'Low')

def flagged_features(fmap):
    flags=[]
    for n,v in fmap.items():
        if n not in FEATURE_MEANS: continue
        d=abs(v-FEATURE_MEANS[n])/(FEATURE_STDS[n]+1e-9)
        if d>=1.3:
            flags.append({'feature':n,'value':float(v),
                'comparison':('higher than' if v>FEATURE_MEANS[n] else 'lower than')+' average',
                'deviation':float(round(d,2))})
    return flags[:6]

def build_prediction_payload(model_name, features):
    scaled=normalize_features(features)
    mdl=MODELS[model_name]
    proba=mdl.predict_proba(scaled)[0]
    pred=int(mdl.predict(scaled)[0])
    label=TARGET_LABELS[pred]
    prob=float(round(proba[1] if pred==1 else 1-proba[1],4))
    risk=risk_level(1-proba[1] if pred==0 else proba[1])
    fmap={n:float(features[i]) for i,n in enumerate(FEATURE_NAMES)}
    return {'model':model_name,'label':label,'prediction':pred,
            'probabilities':{'Benign':float(round(proba[1],4)),'Malignant':float(round(proba[0],4))},
            'confidence':float(round(max(proba)*100,2)),'risk_level':risk,
            'flagged_features':flagged_features(fmap),
            'analysis':agent.generate_report(label,prob,risk,fmap)}

def preprocess_image(image_bytes, size=64, channels=1):
    mode='L' if channels==1 else 'RGB'
    img=Image.open(BytesIO(image_bytes)).convert(mode).resize((size,size),Image.LANCZOS)
    arr=np.asarray(img,dtype=np.float32)/255.0
    arr=arr.reshape(1,size,size,channels)
    th=img.resize((128,128),Image.LANCZOS)
    buf=BytesIO(); th.save(buf,format='PNG')
    prev=base64.b64encode(buf.getvalue()).decode('utf-8')
    return arr, prev

print('✅ Helper functions ready!')

In [ ]:
@app.route('/')
def home(): return render_template('index.html')

@app.route('/api/health')
def health(): return jsonify({'status':'ok','message':'Breast Cancer AI API running.'})

@app.route('/api/feature_names')
def feature_names():
    return jsonify({'feature_names':list(FEATURE_NAMES),'feature_examples':FEATURE_EXAMPLES})

@app.route('/api/dataset_stats')
def get_dataset_stats():
    counts=DF['target'].value_counts().to_dict()
    return jsonify({'samples':int(DF.shape[0]),'features':len(FEATURE_NAMES),
        'classes':list(DATA.target_names),
        'distribution':{TARGET_LABELS[k]:int(v) for k,v in counts.items()},
        'feature_groups':{'mean':[n for n in FEATURE_NAMES if 'mean' in n],
            'se':[n for n in FEATURE_NAMES if 'se' in n],
            'worst':[n for n in FEATURE_NAMES if 'worst' in n]}})

@app.route('/api/model_results')
def get_model_results(): return jsonify(MODEL_RESULTS)

@app.route('/api/compare_models')
def compare_models():
    return jsonify({'models':sorted(MODEL_RESULTS['models'],key=lambda x:x['accuracy'],reverse=True)})

@app.route('/api/cnn_metrics')
def cnn_metrics_route(): return jsonify(CNN_METRICS)

@app.route('/api/multimodal_metrics')
def multimodal_metrics_route(): return jsonify(MULTIMODAL_METRICS)

@app.route('/api/predict', methods=['POST'])
def predict():
    p=request.get_json(force=True)
    mn=p.get('model') or 'Random Forest'
    ft=p.get('features')
    if not ft or len(ft)!=len(FEATURE_NAMES):
        return jsonify({'error':'features must be a list of 30 values'}),400
    try: return jsonify(build_prediction_payload(mn,ft))
    except Exception as e: return jsonify({'error':str(e)}),400

@app.route('/api/predict_image', methods=['POST'])
def predict_image():
    if 'image' not in request.files:
        return jsonify({'error':'No image file provided.'}),400
    try:
        content=request.files['image'].read()
        mdl=lazy_load_cnn()
        arr,prev=preprocess_image(content,64,1)
        proba=float(mdl.predict(arr,verbose=0)[0][0])
        label='Benign' if proba>=0.5 else 'Malignant'
        risk=risk_level(proba)
        return jsonify({'label':label,'probability':round(proba,4),
            'confidence':round(max(proba,1-proba)*100,2),'risk_level':risk,
            'probabilities':{'Benign':round(proba,4),'Malignant':round(1-proba,4)},
            'analysis':agent.generate_report(label,round(proba,4),risk,{}),
            'thumbnail':f'data:image/png;base64,{prev}'})
    except Exception as e: return jsonify({'error':str(e)}),500

@app.route('/api/predict_multimodal', methods=['POST'])
def predict_multimodal():
    ZERO=np.zeros((1,224,224,3),dtype=np.float32)
    inputs,previews=[],[]
    previews_dict={}
    for modality in ['ultrasound','histopathology','chest_xray']:
        f=request.files.get(modality)
        if f and f.filename:
            arr,prev=preprocess_image(f.read(),224,3)
            inputs.append(arr)
            previews_dict[modality]=f'data:image/png;base64,{prev}'
        else:
            inputs.append(ZERO)
            previews_dict[modality]=None
    if all(np.array_equal(x,ZERO) for x in inputs):
        return jsonify({'error':'Provide at least one image.'}),400
    try:
        mdl=lazy_load_multimodal()
        proba=float(mdl.predict(inputs,verbose=0)[0][0])
        label='Benign' if proba>=0.5 else 'Malignant'
        risk=risk_level(proba)
        return jsonify({'label':label,'probability':round(proba,4),
            'confidence':round(max(proba,1-proba)*100,2),'risk_level':risk,
            'probabilities':{'Benign':round(proba,4),'Malignant':round(1-proba,4)},
            'modalities_provided':[m for m in ['ultrasound','histopathology','chest_xray'] if previews_dict[m]],
            'thumbnails':previews_dict,
            'analysis':agent.generate_report(label,round(proba,4),risk,{}),
            'model':'Multimodal EfficientNetB0 (US + Histo + CXR)'})
    except FileNotFoundError as e: return jsonify({'error':str(e)}),503
    except Exception as e: return jsonify({'error':str(e)}),500

@app.route('/api/predict_sample')
def predict_sample():
    idx=random.randint(0,DF.shape[0]-1)
    s=DF.iloc[idx]; ft=s[FEATURE_NAMES].tolist()
    res={'sample_index':int(idx),'features':{n:float(s[n]) for n in FEATURE_NAMES},
         'target':TARGET_LABELS[int(s['target'])],
         'predictions':[build_prediction_payload(n,ft) for n in MODELS]}
    res['agent_report']=agent.generate_report_for_sample(res)
    return jsonify(res)

@app.route('/api/agent/chat', methods=['POST'])
def agent_chat():
    p=request.get_json(force=True)
    q=p.get('question','').strip()
    if not q: return jsonify({'error':'Question required.'}),400
    return jsonify({'question':q,'answer':agent.chat(q,p.get('context',{}))})

@app.errorhandler(413)
def too_large(e): return jsonify({'error':'File too large. Max 16MB.'}),413

print('✅ All routes registered!')

## Step 9 — Launch App with Ngrok
⚠️ Apna ngrok token daalo neeche: **dashboard.ngrok.com → Your Authtoken**

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token('YOUR_NGROK_TOKEN_HERE')  # ← Apna token yahan
public_url = ngrok.connect(5000)
print('🌐 App URL:', public_url)
print('📡 Multimodal API:', str(public_url).split('"')[1] + '/api/predict_multimodal')

In [ ]:
app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)